## New Data Backtesting

New price data from 3/1/26 to 3/31/26 will be retrieved for model backtesting

In [1]:
# import libraries
from datetime import datetime, timezone
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from dotenv import load_dotenv
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pickle
import math
import torch
import torch.nn as nn
import mplfinance as mpf

### Retrieve Price Data

**Only Run Once or to retrieve new price data**

In [2]:
# load alpaca API setup
load_dotenv()
ALPACA_API_KEY    = os.getenv("ALPACA_API_KEY")
ALPACA_SECRET_KEY = os.getenv("ALPACA_SECRET_KEY")

if not ALPACA_API_KEY or not ALPACA_SECRET_KEY:
    raise ValueError("Missing Alpaca keys. Check your .env file.")

client = StockHistoricalDataClient(ALPACA_API_KEY, ALPACA_SECRET_KEY)

In [3]:
# retrieve new data for TSLA, NVDA, OKLO, COIN, PLTR
SYMBOLS = ['COIN', 'NVDA', 'OKLO', 'PLTR', 'TSLA']
START_DATE = datetime(2026, 3, 1,  tzinfo=timezone.utc)
END_DATE = datetime(2026, 3, 31, 23, 59, tzinfo=timezone.utc)

all_dfs = []

for symbol in SYMBOLS:
    req = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=TimeFrame(5, TimeFrameUnit.Minute),
        start=START_DATE,
        end=END_DATE,
        limit=10000,
    )
    bars = client.get_stock_bars(req).df.reset_index()

    if not bars.empty:
        all_dfs.append(bars)
        print(f"{symbol}: {len(bars):,} bars")
    else:
        print(f"{symbol}: NO DATA")

df_raw = pd.concat(all_dfs).reset_index(drop=True)

print(f"\nTotal raw bars: {len(df_raw):,}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"Date range: {df_raw['timestamp'].min()} → {df_raw['timestamp'].max()}")

COIN: 4,157 bars
NVDA: 4,223 bars
OKLO: 3,762 bars
PLTR: 4,158 bars
TSLA: 4,223 bars

Total raw bars: 20,523
Columns: ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']
Date range: 2026-03-02 09:00:00+00:00 → 2026-03-31 23:55:00+00:00


In [4]:
# save csv file
df_raw.to_csv("multiasset_unseen_raw_data.csv", index=False)
print("multiasset_unseen_raw_data.csv")

multiasset_unseen_raw_data.csv


### Regular Trading Hours filtering and Feature Engineering

In [5]:
# call csv file on future runs
df_raw = pd.read_csv('multiasset_unseen_raw_data.csv')

In [6]:
df_raw.head(10)

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,COIN,2026-03-02 09:00:00+00:00,170.60,171.70,170.60,171.12,3875.0,244.0,171.045949
1,COIN,2026-03-02 09:05:00+00:00,171.08,172.42,171.04,172.42,6062.0,124.0,171.554060
2,COIN,2026-03-02 09:10:00+00:00,172.50,173.16,172.50,172.89,930.0,56.0,172.780520
3,COIN,2026-03-02 09:15:00+00:00,172.83,172.89,172.49,172.89,2237.0,79.0,172.725045
4,COIN,2026-03-02 09:20:00+00:00,172.48,173.08,172.20,173.08,4232.0,102.0,172.789183
5,COIN,2026-03-02 09:25:00+00:00,172.81,172.90,171.99,172.19,4977.0,203.0,172.489524
6,COIN,2026-03-02 09:30:00+00:00,172.69,172.76,172.69,172.76,383.0,11.0,172.709296
7,COIN,2026-03-02 09:35:00+00:00,172.35,172.37,172.22,172.37,953.0,29.0,172.335826
8,COIN,2026-03-02 09:40:00+00:00,171.99,172.00,171.99,171.99,1022.0,35.0,171.996751
9,COIN,2026-03-02 09:45:00+00:00,172.15,172.15,171.75,171.75,572.0,27.0,171.833588


In [7]:
# handle timezones
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], utc=True)
df_raw['ts_ny'] = df_raw['timestamp'].dt.tz_convert('America/New_York')
df_raw['date_ny'] = pd.to_datetime(df_raw['ts_ny'].dt.date)

# rth filtering
rth_mask = (
    (df_raw['ts_ny'].dt.time >= pd.Timestamp('09:30').time()) &
    (df_raw['ts_ny'].dt.time <= pd.Timestamp('15:55').time())
)
df = df_raw[rth_mask].copy().reset_index(drop=True)

print(f"Raw bars: {len(df_raw):,}")
print(f"RTH bars: {len(df):,}")
print(f"\nRTH rows per symbol:")
print(df['symbol'].value_counts())

Raw bars: 20,523
RTH bars: 8,580

RTH rows per symbol:
symbol
COIN    1716
NVDA    1716
OKLO    1716
PLTR    1716
TSLA    1716
Name: count, dtype: int64


In [8]:
SYMBOLS = ['COIN', 'NVDA', 'OKLO', 'PLTR', 'TSLA']
# feature engineering
df = df.sort_values(['symbol', 'ts_ny']).reset_index(drop=True)

# session vwap
df['tp'] = (df['high'] + df['low'] + df['close']) / 3
df['cum_tp_vol'] = df.groupby(['symbol', 'date_ny'])['tp'].transform(lambda x: (x * df.loc[x.index, 'volume']).cumsum())
df['cum_vol'] = df.groupby(['symbol', 'date_ny'])['volume'].transform('cumsum')
df['vwap_session'] = df['cum_tp_vol'] / df['cum_vol']
df.drop(columns=['tp', 'cum_tp_vol', 'cum_vol'], inplace=True)

# atr
df['prev_close'] = df.groupby('symbol')['close'].shift(1)
df['tr']         = df[['high', 'low', 'prev_close']].apply(
    lambda r: max(r['high'] - r['low'],
                  abs(r['high'] - r['prev_close']),
                  abs(r['low']  - r['prev_close'])), axis=1
)
df['atr_18']     = df.groupby('symbol')['tr'].transform(lambda x: x.ewm(span=18, adjust=False).mean())
df['atr_pct_18'] = df['atr_18'] / df['close']
df.drop(columns=['prev_close', 'tr'], inplace=True)

# log returns
for n in [1, 3, 6, 12, 18, 24]:
    df[f'ret_{n}'] = df.groupby('symbol')['close'].transform(
        lambda x: np.log(x / x.shift(n))
    )

# vwap distance
df['vwap_dist']     = (df['close'] - df['vwap_session']) / df['close']
df['vwap_dist_atr'] = (df['close'] - df['vwap_session']) / df['atr_18']

# volume impulse
df['vol_ratio_24'] = df.groupby('symbol')['volume'].transform(
    lambda x: x / x.rolling(24, min_periods=1).mean()
)

# realized volatility
df['rv_18'] = df.groupby('symbol')['ret_1'].transform(
    lambda x: x.rolling(18, min_periods=1).std()
)

# range
df['range_pct']      = (df['high'] - df['low'])   / df['close']
df['body_pct']       = abs(df['close'] - df['open']) / df['close']
df['upper_wick_pct'] = (df['high'] - df[['open', 'close']].max(axis=1)) / df['close']
df['lower_wick_pct'] = (df[['open', 'close']].min(axis=1) - df['low'])  / df['close']

# time of day
session_minutes     = (df['ts_ny'].dt.hour * 60 + df['ts_ny'].dt.minute - 570)
session_total       = 390
df['tod_sin']       = np.sin(2 * np.pi * session_minutes / session_total)
df['tod_cos']       = np.cos(2 * np.pi * session_minutes / session_total)

# symbol id
df['symbol_id'] = df['symbol'].map({s: i for i, s in enumerate(SYMBOLS)})

print(f"NaN counts:\n{df[['atr_18', 'ret_24', 'vol_ratio_24', 'rv_18']].isna().sum()}")

NaN counts:
atr_18            0
ret_24          120
vol_ratio_24      0
rv_18            10
dtype: int64


### VWAP Reclaim Trade Identification

1. Prior bar closed below VWAP
2. Current Bar closes above VWAP
3. Volume inpulse >= 1.5X 24-bar average
4. Positive 5-minute return on signal bar

Entry executes at the open of the next bar.

In [9]:
# VWAP Reclaim Long
VOL_IMPULSE = 1.5

df = df.sort_values(['symbol', 'ts_ny']).reset_index(drop=True)

prev_close = df.groupby('symbol')['close'].shift(1)
prev_vwap  = df.groupby('symbol')['vwap_session'].shift(1)

signal_mask = (
    (prev_close          <  prev_vwap)        &
    (df['close']         >  df['vwap_session']) &
    (df['vol_ratio_24']  >= VOL_IMPULSE)       &
    (df['ret_1']         >  0)
)

signal_indices = df[signal_mask].index.tolist()

In [10]:
# create VWAP Reclaim entries
entries_list = []

for i_signal in signal_indices:
    i_entry = i_signal + 1

    if i_entry >= len(df):
        continue

    signal_row = df.iloc[i_signal]
    entry_row  = df.iloc[i_entry]

    # ensure same stock on same session
    if entry_row['symbol'] != signal_row['symbol']:
        continue
    if entry_row['date_ny'] != signal_row['date_ny']:
        continue

    entries_list.append({
        'i_signal': i_signal,
        'i_entry': i_entry,
        'symbol': signal_row['symbol'],
        'date_ny': signal_row['date_ny'],
        'ts_signal': signal_row['ts_ny'],
        'ts_entry': entry_row['ts_ny'],
        'entry_price': entry_row['open'],
        'entry_atr': entry_row['atr_18'],
        'side': 'LONG',
        'timestamp': signal_row['timestamp'],
    })

entries = pd.DataFrame(entries_list).reset_index(drop=True)

print(f"VWAP Reclaim setups identified: {len(entries)}")
print(f"\nBy symbol:")
print(entries['symbol'].value_counts())
print(f"\nDate range: {entries['date_ny'].min().date()} → {entries['date_ny'].max().date()}")

VWAP Reclaim setups identified: 71

By symbol:
symbol
PLTR    17
NVDA    16
COIN    14
TSLA    14
OKLO    10
Name: count, dtype: int64

Date range: 2026-03-02 → 2026-03-31


### Follow Through Analysis

FT_THRESHOLD = 2.5 ATR

In [11]:
# calculate MFE and MAE for follow-through
FORWARD_BARS = 24
FT_THRESHOLD = 2.5
SCALP_MODE = {"stop_atr": 0.5, "target_atr": 0.75, "hold_bars": 6, "risk_per_trade": 750}
HOLD_MODE = {"stop_atr": 1.0, "target_atr": 2.5, "hold_bars": 24, "risk_per_trade": 1500}

def compute_mfe_mae(df, i_entry, forward_bars=FORWARD_BARS):
    entry_row = df.iloc[i_entry]
    entry_price = entry_row['open']
    atr = entry_row['atr_18']
    symbol = entry_row['symbol']
    date = entry_row['date_ny']

    if pd.isna(atr) or atr <= 0:
        return None

    window = df.iloc[i_entry : i_entry + forward_bars + 1]
    window = window[(window['symbol'] == symbol) & (window['date_ny'] == date)]

    if len(window) < 2:
        return None

    highs = window['high'].values
    lows = window['low'].values

    return {
        'mfe_atr': round((highs.max() - entry_price) / atr, 4),
        'mae_atr': round((entry_price - lows.min())  / atr, 4),
        'time_to_peak_bars': int(highs.argmax())
    }

In [12]:
# simulate trades
def simulate_trade(df, i_entry, stop_atr, target_atr, hold_bars, **kwargs):
    entry_row = df.iloc[i_entry]
    entry_price = entry_row['open']
    atr = entry_row['atr_18']
    symbol = entry_row['symbol']
    date = entry_row['date_ny']

    if pd.isna(atr) or atr <= 0:
        return None

    risk = stop_atr * atr
    stop_price = entry_price - risk
    target_price = entry_price + target_atr * atr

    window = df.iloc[i_entry : i_entry + hold_bars + 1]
    window = window[(window['symbol'] == symbol) & (window['date_ny'] == date)]

    exit_price, exit_reason = None, None
    for _, bar in window.iterrows():
        if bar['low'] <= stop_price:
            exit_price, exit_reason = stop_price, 'STOP'; break
        elif bar['high'] >= target_price:
            exit_price, exit_reason = target_price, 'TARGET'; break

    if exit_price is None:
        exit_price = window.iloc[-1]['close']
        exit_reason = 'TIME'

    R = (exit_price - entry_price) / risk
    return {'R': round(R, 4), 'exit_reason': exit_reason}

In [13]:
# run on all vwap reclaim long trades
mfe_list, scalp_list, hold_list = [], [], []

for _, row in entries.iterrows():
    i = int(row['i_entry'])
    mfe_list.append(compute_mfe_mae(df, i) or
        {'mfe_atr': None, 'mae_atr': None, 'time_to_peak_bars': None})
    scalp_list.append(simulate_trade(df, i, **SCALP_MODE) or
        {'R': None, 'exit_reason': None})
    hold_list.append(simulate_trade(df, i, **HOLD_MODE) or
        {'R': None, 'exit_reason': None})

mfe_df = pd.DataFrame(mfe_list)
scalp_df = pd.DataFrame(scalp_list).add_prefix('scalp_')
hold_df = pd.DataFrame(hold_list).add_prefix('hold_')

entries[['mfe_atr', 'mae_atr', 'time_to_peak_bars']] = mfe_df[['mfe_atr', 'mae_atr', 'time_to_peak_bars']]
entries[['R_scalp', 'exit_reason_scalp']] = scalp_df[['scalp_R', 'scalp_exit_reason']]
entries[['R_hold', 'exit_reason_hold']] = hold_df[['hold_R', 'hold_exit_reason']]
entries = entries.dropna(subset=['mfe_atr', 'R_scalp', 'R_hold']).reset_index(drop=True)
# Follow-through label
entries['follow_through'] = (entries['mfe_atr'] >= FT_THRESHOLD).astype(int)

# Oracle
entries['R_oracle'] = entries.apply(lambda r: r['R_hold']  if r['follow_through'] == 1 else r['R_scalp'], axis=1)
entries['risk_oracle'] = entries.apply(lambda r: HOLD_MODE['risk_per_trade'] if r['follow_through'] == 1 else SCALP_MODE['risk_per_trade'], axis=1)

print(f"Follow-Through (≥2.5 ATR): {entries['follow_through'].sum()} / {len(entries)} ({entries['follow_through'].mean():.1%})")
print(f"\nMFE distribution:")
print(entries['mfe_atr'].describe().round(3))
print(f"\nOracle P&L ceiling: ${(entries['R_oracle'] * entries['risk_oracle']).sum():,.0f}")

Follow-Through (≥2.5 ATR): 20 / 68 (29.4%)

MFE distribution:
count    68.000
mean      1.833
std       1.252
min       0.038
25%       0.789
50%       1.582
75%       2.667
max       4.956
Name: mfe_atr, dtype: float64

Oracle P&L ceiling: $41,250


### Load Model for Predictions

In [14]:
# load scaler features
with open('feature_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

FEATURE_COLS = scaler['feature_cols']
feat_mean = scaler['mean']
feat_std = scaler['std']
N_FEATURES = len(FEATURE_COLS)

SEQ_LEN = 48
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 2
D_FF = 256
DROPOUT = 0.1
SYMBOL_EMB = 8
INPUT_DIM = N_FEATURES + SYMBOL_EMB
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ENTRY_FEAT_COLS = [
    'tod_sin',
    'tod_cos',
    'vwap_dist_atr',
    'vol_ratio_24',
    'atr_pct_18',
    'ret_1',
]
N_ENTRY_FEATURES = len(ENTRY_FEAT_COLS)

In [15]:
# merge entry level features to entries
entry_feat_merge = df.iloc[entries['i_signal'].values][ENTRY_FEAT_COLS].reset_index(drop=True)
entries = pd.concat([entries.reset_index(drop=True), entry_feat_merge], axis=1)

In [16]:
# define transformer model
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# core encoder
class BarTransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.symbol_embedding = nn.Embedding(len(SYMBOLS), SYMBOL_EMB)
        self.input_proj = nn.Linear(INPUT_DIM, D_MODEL)
        self.pos_encoding = SinusoidalPositionalEncoding(D_MODEL)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            dropout=DROPOUT, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=N_LAYERS, enable_nested_tensor=False
        )
        self.norm = nn.LayerNorm(D_MODEL)
    def forward(self, x, symbol_id):
        sym_emb = self.symbol_embedding(symbol_id)
        sym_emb = sym_emb.unsqueeze(1).expand(-1, x.size(1), -1)
        x = torch.cat([x, sym_emb], dim=-1)
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        x = self.transformer(x)
        return self.norm(x)

# follow through classifier
class FollowThroughClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = BarTransformerEncoder()
        self.classifier = nn.Sequential(
            nn.Linear(D_MODEL + N_ENTRY_FEATURES, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )
    def forward(self, x, symbol_id, entry_feats):
        enc_out = self.encoder(x, symbol_id)
        pooled = enc_out.mean(dim=1)
        combined = torch.cat([pooled, entry_feats], dim=-1)
        return self.classifier(combined).squeeze(-1)

model = FollowThroughClassifier().to(DEVICE)
model.load_state_dict(torch.load('follow_through_classifier.pt', map_location=DEVICE))
model.eval()

FollowThroughClassifier(
  (encoder): BarTransformerEncoder(
    (symbol_embedding): Embedding(5, 8)
    (input_proj): Linear(in_features=24, out_features=64, bias=True)
    (pos_encoding): SinusoidalPositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      

In [17]:
# extract sequences
def extract_sequence(df, i_entry, symbol, date_ny):
    start = max(0, i_entry - SEQ_LEN - 10)
    window = df.iloc[start:i_entry]
    window = window[(window['symbol'] == symbol) & (window['date_ny'] == date_ny)]
    window = window.tail(SEQ_LEN)
    feats = window[FEATURE_COLS].values.astype(np.float32)
    feats = (feats - feat_mean.values) / feat_std.values
    feats = np.nan_to_num(feats, nan=0.0)
    feats = np.clip(feats, -5.0, 5.0)
    if len(feats) < SEQ_LEN:
        pad = np.zeros((SEQ_LEN - len(feats), N_FEATURES), dtype=np.float32)
        feats = np.vstack([pad, feats])
    return feats

OPERATING_THRESHOLD = 0.55
sequences, symbol_ids, entry_feats_list = [], [], []

# extract entry features
for _, row in entries.iterrows():
    seq = extract_sequence(df, int(row['i_entry']), row['symbol'], row['date_ny'])
    sequences.append(seq)
    symbol_ids.append(SYMBOLS.index(row['symbol']))

    ef = [(row[c] - feat_mean[c]) / feat_std[c] if c in feat_mean.index else row[c]
          for c in ENTRY_FEAT_COLS]
    ef = np.clip(np.nan_to_num(ef, nan=0.0), -5.0, 5.0)
    entry_feats_list.append(ef)

sequences = torch.tensor(np.array(sequences), dtype=torch.float32).to(DEVICE)
symbol_ids = torch.tensor(np.array(symbol_ids), dtype=torch.int64).to(DEVICE)
entry_feats_list = torch.tensor(np.array(entry_feats_list), dtype=torch.float32).to(DEVICE)

In [18]:
# make predictions
with torch.no_grad():
    logits = model(sequences, symbol_ids, entry_feats_list)
    probs = torch.sigmoid(logits).cpu().numpy()

entries['p_follow_through'] = probs
entries['pred_follow_through'] = (probs >= OPERATING_THRESHOLD).astype(int)

print(f"\nOperating threshold: {OPERATING_THRESHOLD}")
print(f"Predicted Hold: {entries['pred_follow_through'].sum()} / {len(entries)} ({entries['pred_follow_through'].mean():.1%})")
print(f"Actual Follow-Through: {entries['follow_through'].sum()} / {len(entries)} ({entries['follow_through'].mean():.1%})")
print(f"\nProbability distribution:")
print(pd.Series(probs).describe().round(3))


Operating threshold: 0.55
Predicted Hold: 12 / 68 (17.6%)
Actual Follow-Through: 20 / 68 (29.4%)

Probability distribution:
count    68.000
mean      0.365
std       0.204
min       0.087
25%       0.221
50%       0.298
75%       0.406
max       0.968
dtype: float64


## Trade Backtesting

The following strategies were backtested:
- Scalp Only: scalp every trade
- Hold only if predicted: only trade when model predicts follow through
- Adaptive: hold if model predicts follow through, else scalp
- Oracle: Use simulation results from best possible outcomes

In [19]:
SCALP_RISK = 750
HOLD_RISK  = 1500

# scalp only
entries['pnl_scalp'] = entries['R_scalp'] * SCALP_RISK

# adaptive
entries['R_adaptive'] = entries.apply(
    lambda r: r['R_hold'] if r['pred_follow_through'] == 1 else r['R_scalp'],
    axis=1
)
entries['risk_adaptive'] = entries['pred_follow_through'].map({1: HOLD_RISK, 0: SCALP_RISK})
entries['pnl_adaptive'] = entries['R_adaptive'] * entries['risk_adaptive']

# hold if predicted
entries['pnl_hold_only'] = entries.apply(
    lambda r: r['R_hold'] * HOLD_RISK if r['pred_follow_through'] == 1 else 0,
    axis=1
)

# oracle
entries['pnl_oracle'] = entries['R_oracle'] * entries['risk_oracle']

# cumulative P&L
entries = entries.sort_values('date_ny').reset_index(drop=True)
entries['cum_pnl_scalp'] = entries['pnl_scalp'].cumsum()
entries['cum_pnl_adaptive'] = entries['pnl_adaptive'].cumsum()
entries['cum_pnl_hold_only'] = entries['pnl_hold_only'].cumsum()
entries['cum_pnl_oracle'] = entries['pnl_oracle'].cumsum()

# summary stats
def summary_row(pnl_col, r_col, label):
    pnl = entries[pnl_col]
    r = entries[r_col]

    traded = pnl[pnl != 0]
    win_rate = (traded > 0).mean() if len(traded) > 0 else 0

    # daily Sharpe for consistency
    daily_pnl = entries.groupby('date_ny')[pnl_col].sum()
    sharpe = daily_pnl.mean() / daily_pnl.std() * np.sqrt(252) if daily_pnl.std() > 0 else 0

    return {
        'Strategy': label,
        'Total P&L': pnl.sum(),
        'Win Rate': win_rate,
        'Avg R': r.mean(),
        'Daily Sharpe': sharpe
    }

summary_table = pd.DataFrame([
    summary_row('pnl_scalp', 'R_scalp', 'Scalp (always Scalp)'),
    summary_row('pnl_adaptive', 'R_adaptive', 'Adaptive (model-routed)'),
    summary_row('pnl_hold_only', 'R_hold', 'Hold if Predicted'),
    summary_row('pnl_oracle', 'R_oracle', 'Oracle')
]).set_index('Strategy')

summary_table_display = summary_table.copy()
summary_table_display['Total P&L'] = summary_table_display['Total P&L'].map(lambda x: f"${x:,.0f}")
summary_table_display['Win Rate'] = summary_table_display['Win Rate'].map(lambda x: f"{x:.1%}")
summary_table_display['Avg R'] = summary_table_display['Avg R'].map(lambda x: f"{x:.3f}")
summary_table_display['Daily Sharpe'] = summary_table_display['Daily Sharpe'].map(lambda x: f"{x:.3f}")

print("BACKTEST MARCH 2026")
print(summary_table_display)

fixed_pnl = entries['pnl_scalp'].sum()
adaptive_pnl = entries['pnl_adaptive'].sum()
hold_only_pnl = entries['pnl_hold_only'].sum()
oracle_pnl = entries['pnl_oracle'].sum()

comparison_table = pd.DataFrame([
    ['Oracle Ceiling', oracle_pnl],
    ['Adaptive Gain vs Scalp', adaptive_pnl - fixed_pnl],
    ['Hold-if-Predicted Gain vs Scalp', hold_only_pnl - fixed_pnl]
], columns=['Metric', 'Value']).set_index('Metric')

comparison_table_display = comparison_table.copy()
comparison_table_display['Value'] = comparison_table_display['Value'].map(lambda x: f"${x:,.0f}")

print("\nCOMPARISON")
print(comparison_table_display)

BACKTEST MARCH 2026
                        Total P&L Win Rate   Avg R Daily Sharpe
Strategy                                                       
Scalp (always Scalp)       $5,250    44.1%   0.103        2.086
Adaptive (model-routed)   $11,396    45.6%   0.178        2.969
Hold if Predicted          $4,646    41.7%  -0.098        1.583
Oracle                    $41,250    47.1%   0.382        8.775

COMPARISON
                                   Value
Metric                                  
Oracle Ceiling                   $41,250
Adaptive Gain vs Scalp            $6,146
Hold-if-Predicted Gain vs Scalp    $-604


In [20]:
# adaptive strategy P&L breakdown
tp = entries[(entries['pred_follow_through']==1) & (entries['follow_through']==1)]
fp = entries[(entries['pred_follow_through']==1) & (entries['follow_through']==0)]
tn = entries[(entries['pred_follow_through']==0) & (entries['follow_through']==0)]
fn = entries[(entries['pred_follow_through']==0) & (entries['follow_through']==1)]

adaptive_confusion_pnl = pd.DataFrame([
    ['TP', 'Pred Hold', 'Actual Follow-Through', 'Trade Hold', len(tp), tp['pnl_adaptive'].sum(), tp['pnl_adaptive'].mean()],
    ['FP', 'Pred Hold', 'Actual No Follow-Through', 'Trade Hold', len(fp), fp['pnl_adaptive'].sum(), fp['pnl_adaptive'].mean()],
    ['TN', 'Pred Scalp', 'Actual No Follow-Through', 'Trade Scalp', len(tn), tn['pnl_adaptive'].sum(), tn['pnl_adaptive'].mean()],
    ['FN', 'Pred Scalp', 'Actual Follow-Through', 'Trade Scalp', len(fn), fn['pnl_adaptive'].sum(), fn['pnl_adaptive'].mean()],
], columns=[
    'Bucket', 'Prediction', 'Actual Outcome', 'Action', 'Count', 'Total P&L', 'Avg P&L'
])

adaptive_confusion_pnl['Total P&L'] = adaptive_confusion_pnl['Total P&L'].map(lambda x: f"${x:,.0f}")
adaptive_confusion_pnl['Avg P&L'] = adaptive_confusion_pnl['Avg P&L'].map(lambda x: f"${x:,.0f}")

print("ADAPTIVE - CONFUSION-WEIGHTED P&L MATRIX")
print(adaptive_confusion_pnl)

print(f"\nNet Adaptive P&L:${entries['pnl_adaptive'].sum():,.0f}")

ADAPTIVE - CONFUSION-WEIGHTED P&L MATRIX
  Bucket  Prediction            Actual Outcome       Action  Count Total P&L  \
0     TP   Pred Hold     Actual Follow-Through   Trade Hold      4    $9,750   
1     FP   Pred Hold  Actual No Follow-Through   Trade Hold      8   $-5,104   
2     TN  Pred Scalp  Actual No Follow-Through  Trade Scalp     40        $0   
3     FN  Pred Scalp     Actual Follow-Through  Trade Scalp     16    $6,750   

  Avg P&L  
0  $2,438  
1   $-638  
2      $0  
3    $422  

Net Adaptive P&L:$11,396


In [21]:
# Hold if predicted only P&L breakdown
hold_confusion_pnl = pd.DataFrame([
    ['TP', 'Pred Hold', 'Actual Follow-Through', 'Trade Hold', len(tp), tp['pnl_hold_only'].sum(), tp['pnl_hold_only'].mean()],
    ['FP', 'Pred Hold', 'Actual No Follow-Through', 'Trade Hold', len(fp), fp['pnl_hold_only'].sum(), fp['pnl_hold_only'].mean()],
    ['TN', 'Pred Scalp', 'Actual No Follow-Through', 'Skip Trade', len(tn), tn['pnl_hold_only'].sum(), tn['pnl_hold_only'].mean()],
    ['FN', 'Pred Scalp', 'Actual Follow-Through', 'Skip Trade', len(fn), fn['pnl_hold_only'].sum(), fn['pnl_hold_only'].mean()],
], columns=[
    'Bucket', 'Prediction', 'Actual Outcome', 'Action', 'Count', 'Total P&L', 'Avg P&L'
])

hold_confusion_pnl['Total P&L'] = hold_confusion_pnl['Total P&L'].map(lambda x: f"${x:,.0f}")
hold_confusion_pnl['Avg P&L'] = hold_confusion_pnl['Avg P&L'].map(lambda x: f"${x:,.0f}")

print("HOLD IF PREDICTED - CONFUSION-WEIGHTED P&L MATRIX")
print(hold_confusion_pnl)

print(f"\nNet Hold if Predicted P&L:${entries['pnl_hold_only'].sum():,.0f}")

HOLD IF PREDICTED - CONFUSION-WEIGHTED P&L MATRIX
  Bucket  Prediction            Actual Outcome      Action  Count Total P&L  \
0     TP   Pred Hold     Actual Follow-Through  Trade Hold      4    $9,750   
1     FP   Pred Hold  Actual No Follow-Through  Trade Hold      8   $-5,104   
2     TN  Pred Scalp  Actual No Follow-Through  Skip Trade     40        $0   
3     FN  Pred Scalp     Actual Follow-Through  Skip Trade     16        $0   

  Avg P&L  
0  $2,438  
1   $-638  
2      $0  
3      $0  

Net Hold if Predicted P&L:$4,646


In [22]:
# scalp only
scalp_confusion_pnl = pd.DataFrame([
    ['TP', 'Pred Hold', 'Actual Follow-Through', 'Trade Scalp', len(tp), tp['pnl_scalp'].sum(), tp['pnl_scalp'].mean()],
    ['FP', 'Pred Hold', 'Actual No Follow-Through', 'Trade Scalp', len(fp), fp['pnl_scalp'].sum(), fp['pnl_scalp'].mean()],
    ['TN', 'Pred Scalp', 'Actual No Follow-Through', 'Trade Scalp', len(tn), tn['pnl_scalp'].sum(), tn['pnl_scalp'].mean()],
    ['FN', 'Pred Scalp', 'Actual Follow-Through', 'Trade Scalp', len(fn), fn['pnl_scalp'].sum(), fn['pnl_scalp'].mean()],
], columns=[
    'Bucket', 'Prediction', 'Actual Outcome', 'Action', 'Count', 'Total P&L', 'Avg P&L'
])

scalp_confusion_pnl['Total P&L'] = scalp_confusion_pnl['Total P&L'].map(lambda x: f"${x:,.0f}")
scalp_confusion_pnl['Avg P&L'] = scalp_confusion_pnl['Avg P&L'].map(lambda x: f"${x:,.0f}")

print("SCALP ONLY - CONFUSION-WEIGHTED P&L MATRIX")
print(scalp_confusion_pnl)

print(f"\nNet Scalp Only P&L:${entries['pnl_scalp'].sum():,.0f}")

SCALP ONLY - CONFUSION-WEIGHTED P&L MATRIX
  Bucket  Prediction            Actual Outcome       Action  Count Total P&L  \
0     TP   Pred Hold     Actual Follow-Through  Trade Scalp      4      $750   
1     FP   Pred Hold  Actual No Follow-Through  Trade Scalp      8   $-2,250   
2     TN  Pred Scalp  Actual No Follow-Through  Trade Scalp     40        $0   
3     FN  Pred Scalp     Actual Follow-Through  Trade Scalp     16    $6,750   

  Avg P&L  
0    $188  
1   $-281  
2      $0  
3    $422  

Net Scalp Only P&L:$5,250


In [23]:
# Oracle P&L breakdown
oracle_confusion_pnl = pd.DataFrame([
    ['TP', 'Pred Hold', 'Actual Follow-Through', 'Oracle Choice', len(tp), tp['pnl_oracle'].sum(), tp['pnl_oracle'].mean()],
    ['FP', 'Pred Hold', 'Actual No Follow-Through', 'Oracle Choice', len(fp), fp['pnl_oracle'].sum(), fp['pnl_oracle'].mean()],
    ['TN', 'Pred Scalp', 'Actual No Follow-Through', 'Oracle Choice', len(tn), tn['pnl_oracle'].sum(), tn['pnl_oracle'].mean()],
    ['FN', 'Pred Scalp', 'Actual Follow-Through', 'Oracle Choice', len(fn), fn['pnl_oracle'].sum(), fn['pnl_oracle'].mean()],
], columns=[
    'Bucket', 'Prediction', 'Actual Outcome', 'Action', 'Count', 'Total P&L', 'Avg P&L'
])

oracle_confusion_pnl['Total P&L'] = oracle_confusion_pnl['Total P&L'].map(lambda x: f"${x:,.0f}")
oracle_confusion_pnl['Avg P&L'] = oracle_confusion_pnl['Avg P&L'].map(lambda x: f"${x:,.0f}")

print("ORACLE - CONFUSION-WEIGHTED P&L MATRIX")
print(oracle_confusion_pnl)

print(f"\nNet Oracle P&L:${entries['pnl_oracle'].sum():,.0f}")

ORACLE - CONFUSION-WEIGHTED P&L MATRIX
  Bucket  Prediction            Actual Outcome         Action  Count  \
0     TP   Pred Hold     Actual Follow-Through  Oracle Choice      4   
1     FP   Pred Hold  Actual No Follow-Through  Oracle Choice      8   
2     TN  Pred Scalp  Actual No Follow-Through  Oracle Choice     40   
3     FN  Pred Scalp     Actual Follow-Through  Oracle Choice     16   

  Total P&L Avg P&L  
0    $9,750  $2,438  
1   $-2,250   $-281  
2        $0      $0  
3   $33,750  $2,109  

Net Oracle P&L:$41,250
